# 实验9：CANN 开发环境搭建与模型离线推理体验

## 一、实验目的

本实验基于香橙派AIpro开发板，旨在：

1. 掌握昇腾CANN异构计算架构的基本概念、软件栈组成及其在昇腾AI生态中的核心地位；
2. 能够独立完成CANN开发环境的搭建与配置，包括镜像烧录、CANN软件包安装、环境变量配置等；
3. 熟练使用ATC模型转换工具，理解原始模型文件到昇腾离线模型（.om格式）的转换流程；
4. 完成大语言模型在资源受限的昇腾端侧设备上的部署与离线推理，体验大模型离线处理技术；
5. 理解AscendCL推理应用开发的基本流程，培养在边缘AI设备上进行模型部署与推理的实践能力。


## 二、实验说明

### 2.1 实验背景

随着人工智能技术向边缘侧延伸，大模型在资源受限设备上的部署与推理成为AI领域的重要研究方向。华为昇腾AI生态以CANN异构计算架构为核心，向上支持MindSpore、PyTorch、TensorFlow等多种AI框架，向下适配昇腾系列AI处理器，为开发者提供从模型训练到推理部署的全栈能力。CANN自2018年发布以来历经7年迭代演进，已于2025年8月全面开源开放，实现了组件分层解耦与代码全面开源。

香橙派AIpro开发板是香橙派联合昇腾AI打造的高性能AI开发板，搭载昇腾310/310B4处理器，提供8～20 TOPS的AI算力，广泛应用于AI边缘计算、深度视觉学习、自然语言处理等场景。本实验将以此为硬件平台，完成CANN环境搭建与大模型离线推理的全流程实践。

### 2.2 实验环境准备（**CANNLab在线实验路过这一环节**）

- **硬件环境**：香橙派AIpro开发板（8T/20T版本均可），建议配备32GB以上TF卡用于系统镜像烧录，显示器、HDMI线、键盘、鼠标和电源适配器。
- **操作系统**：Ubuntu 22.04 Desktop镜像（aarch64架构）。
- **软件依赖**：CANN Toolkit 8.0.RC3及以上版本。
- **开发工具**：终端命令行（SSH远程登录或本地操作），可选Jupyter Notebook用于交互式模型推理。
- **网络环境**：首次运行时需网络下载依赖包和模型权重。


## 三、实验任务

### 3.1 任务描述

本实验涵盖以下几个核心任务：

- **任务一**：了解香橙派AIpro开发板的硬件架构与昇腾AI异构计算体系。
- **任务二**：完成CANN开发环境的搭建，包括系统镜像烧录、CANN软件包安装与环境变量配置。
- **任务三**：学习ATC（Ascend Tensor Compiler）模型转换工具的使用方法，将原始模型转换为昇腾离线模型。
- **任务四**：基于CANN和MindSpore/MindIE等框架，完成大语言模型（如Qwen-1.5-0.5B或DeepSeek-R1-Distill-Qwen-1.5B）的部署与离线推理，体验对话交互。


### 3.2 学习目标

通过完成本实验任务，预期达成以下学习目标：

1. **理解昇腾AI全栈架构**：能够描述CANN在昇腾AI生态中的定位与作用，理解从AI框架到AI处理器的端到端计算链路。
2. **具备CANN环境搭建能力**：能够独立完成香橙派AIpro开发板的系统部署与CANN软件栈的安装配置。
3. **掌握模型离线转换技术**：理解ATC工具的工作原理，能够使用atc命令完成ONNX等格式模型到昇腾离线模型的转换。
4. **具备端侧大模型推理开发能力**：能够基于AscendCL或MindSpore完成大模型的加载与推理，理解离线推理应用开发的核心流程。
5. **了解端侧推理的性能优化方向**：对量化、算子融合、内存管理等端侧优化技术有初步认识。

## 四、任务准备

### 4.1 前置知识

- **香橙派AIpro开发板基础：** 香橙派AIpro采用昇腾AI技术路线，搭载4核64位处理器与AI处理器（昇腾310B4），集成DaVinci架构AI Core，AI算力方面，8T版本提供8 TOPS（INT8），20T版本提供20 TOPS（INT8）。开发板配备LPDDR4X内存（8GB/16GB/24GB），支持TF卡、eMMC和NVMe SSD三种启动方式。接口包括双HDMI 2.0、MIPI DSI、双MIPI CSI、USB 3.0、Type-C、双2.5G以太网、M.2插槽等。（**CANNLab在线实验忽略**）

- **CANN架构基础知识：** CANN（Compute Architecture for Neural Networks）是昇腾针对AI场景推出的异构计算架构，向下服务AI处理器，向上支持多种AI框架。CANN通过TBE算子开发工具和AscendCL构建双层抽象：TBE提供1000+预置算子库，AscendCL提供Device管理、Context管理、模型加载与执行等API。在算子开发层面，CANN提供了面向AI Core的Ascend C编程语言，通过多层接口抽象满足高性能算子开发需求。

- **大模型离线推理概念：** 离线推理是指在不依赖网络连接的情况下，利用本地计算资源执行模型推理。在资源受限的端侧设备上部署大模型，通常需要采用模型量化、剪枝、蒸馏等压缩技术，以及通过ATC将模型编译为适配NPU的离线模型格式（.om文件）来实现高效推理。

### 4.2 实验数据准备（**CANNLab在线实验跳过这一环节**）

实验开始前，需要准备以下资源：

1. **系统镜像**：从香橙派官网下载最新的Ubuntu 22.04 Desktop镜像文件（.img.xz格式）。
2. **CANN软件包**：从昇腾社区下载中心获取CANN Toolkit安装包（.run格式，aarch64架构）。
3. **模型文件**：准备待部署的大语言模型权重（实验以Qwen-1.5-0.5B-Chat或DeepSeek-R1-Distill-Qwen-1.5B为例，首次运行时会自动下载）。
4. **烧录工具**：如balenaEtcher，用于将系统镜像写入TF卡。
5. **SSH工具**（可选）：如MobaXterm或VSCode Remote-SSH，用于远程连接开发板。


## 五、任务实施（具体实验步骤，**CANNLab在线实验直接跳到步骤四**）

### 步骤一：硬件准备与确认

（1）检查香橙派AIpro开发板及配件是否齐全，确认TF卡容量充足（建议64GB及以上）。

（2）将开发板背面的拨码开关设置为“右右”模式（TF卡启动），这是推荐新手使用的启动方式。

（3）将TF卡插入读卡器并连接至PC，准备进行镜像烧录。

### 步骤二：系统镜像烧录

（1）启动balenaEtcher工具，选择下载好的Ubuntu 22.04桌面版镜像文件。


（2）选择目标TF卡作为写入设备，点击“Flash”开始烧录。

（3）烧录完成后，将TF卡插入开发板TF卡槽，连接显示器（通过HDMI）、键盘、鼠标和电源适配器。**注意**：建议使用官方电源适配器以确保供电稳定，供电不足可能导致系统无法正常启动。

（4）接通电源后系统自动启动，首次启动时约需1～2分钟完成初始化，随后进入Ubuntu桌面环境。

### 步骤三：CANN环境搭建

香橙派AIpro开发板的官方镜像通常已预装CANN基础版本。若需升级到最新版本，按以下步骤操作。

**（1）切换至root用户**

打开终端（快捷键Ctrl+Alt+T），执行以下命令：

```
su
```

输入root密码后切换到root用户。

**（2）卸载旧版CANN软件包（如已预装）**

进入CANN安装目录，删除旧文件以释放磁盘空间：

```
cd /usr/local/Ascend/ascend-toolkit/
rm -rf *
```

此步骤可防止安装新版CANN时磁盘空间不足。

**（3）下载并安装新版CANN Toolkit**

从昇腾社区下载中心获取最新版CANN Toolkit（选择aarch64架构的.run文件），下载后进入下载目录执行安装：

```
cd /home/HwHiAiUser/Downloads
chmod +x ./Ascend-cann-toolkit-8.0.RC3.alpha002_linux-aarch64.run
./Ascend-cann-toolkit-8.0.RC3.alpha002_linux-aarch64.run --install
```

安装过程中提示确认时输入Y并回车，等待安装完成。

**（4）配置环境变量**

安装完成后，执行以下命令使CANN环境变量生效：

```
source /usr/local/Ascend/ascend-toolkit/set_env.sh
```

建议将该命令追加到/etc/profile或~/.bashrc文件的末尾，以便每次登录时自动生效。

**（5）验证安装**

执行以下命令确认CANN环境配置正确：

```
npu-smi info
```

若显示NPU设备信息（芯片型号、内存、温度等），则说明CANN环境搭建成功。

### 步骤四：ATC模型转换体验

ATC（Ascend Tensor Compiler）是昇腾提供的离线模型转换工具，可将Caffe、TensorFlow、ONNX、MindSpore等主流框架导出的模型文件转换为适配昇腾AI处理器的离线模型（.om格式）。

**（1）准备环境**

首先确认ATC工具所在路径已加入环境变量，可通过以下命令验证：

```
which atc
```

若返回ATC工具的完整路径，则说明环境准备就绪。ATC工具运行前需安装CANN软件包并设置环境变量。

**（2）获取ONNX模型文件**

以ResNet-50为例，从昇腾ModelZoo获取ONNX模型文件，并上传至开发环境的指定目录（如$HOME/module/）。

**（3）执行模型转换**

使用atc命令将ONNX模型转换为昇腾离线模型：

```
atc --model=$HOME/module/resnet50.onnx \
    --framework=5 \
    --output=$HOME/module/out/onnx_resnet50 \
    --soc_version=<soc_version>
```

**参数说明**：

- `--model`：指定原始模型文件路径和文件名；
- `--framework`：原始模型框架类型，`5`代表ONNX模型（`3`=TensorFlow，`1`=MindSpore）；
- `--output`：转换后的离线模型存储路径及文件名前缀，生成的文件自动以.om后缀结尾；
- `--soc_version`：昇腾AI处理器型号，香橙派AIpro对应昇腾310B4，可通过`npu-smi info`查询。

对于需要指定输入格式和尺寸的模型，可添加额外参数：

```
atc --model=model.onnx \
    --framework=5 \
    --output=model \
    --input_format=NCHW \
    --input_shape="input:1,3,224,224" \
    --soc_version=Ascend310B4
```

**关键注意事项**：`--soc_version`必须与实际硬件型号一致，否则生成的离线模型无法加载。

**（4）验证转换结果**

转换成功后，在`--output`指定的路径下可找到生成的.om离线模型文件。可使用atc工具查看模型信息：

```
atc --mode=1 --om=$HOME/module/out/onnx_resnet50.om --json=$HOME/module/out/model_info.json
```

该命令将离线模型的信息导出为JSON文件，便于查看模型的算子组成和资源占用情况。

### 步骤五：大模型离线推理体验（**CANNLab在线实验跳过（1）~（2），直接从（3）开始**）

本步骤以基于香橙派AIpro运行Qwen-1.5-0.5B-Chat模型为例，体验大模型离线推理流程。

**（1）确认版本环境**

实验需要CANN 8.0.RC3版本和MindSpore 2.5.0版本。可通过以下命令检查当前版本：

```
cat /usr/local/Ascend/ascend-toolkit/latest/version.cfg
pip show mindspore
```

**（2）安装依赖包**

在终端中执行以下命令安装必要的Python依赖：

```
pip install git+https://github.com/mindspore-lab/mindnlp.git
```

mindnlp套件包含自然语言处理的常用方法，可方便地加载和使用HuggingFace模型权重。

**（3）加载模型并执行推理**

编写并执行以下代码：

```python
import mindspore
from mindnlp.transformers import AutoTokenizer, AutoModelForCausalLM

# 加载模型和分词器（首次运行会下载模型权重）
model_name = "Qwen/Qwen1.5-0.5B-Chat"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# 对话推理
prompt = "请介绍一下昇腾CANN计算架构的主要特点"
messages = [{"role": "user", "content": prompt}]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="ms")
outputs = model.generate(inputs.input_ids, max_new_tokens=256)
response = tokenizer.decode(outputs[0][len(inputs.input_ids[0]):], skip_special_tokens=True)
print(response)
```

首次运行时，模型权重会自动下载到本地，后续运行可直接使用本地缓存，实现完全离线推理。


In [1]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

import time
from transformers import AutoTokenizer, AutoModelForCausalLM

# 加载模型和分词器（首次运行会下载模型权重）
model_name = "Qwen/Qwen1.5-0.5B-Chat"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# 对话推理
prompt = "请介绍一下昇腾CANN计算架构的主要特点"
messages = [{"role": "user", "content": prompt}]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt")

start_time = time.time()
outputs = model.generate(inputs.input_ids, max_new_tokens=256)
end_time = time.time()
print("Time cost: ", (end_time - start_time) * 100 / 1000, "ms")

response = tokenizer.decode(outputs[0][len(inputs.input_ids[0]):], skip_special_tokens=True)
print(response)

/home/developer/.local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/buildtools/Python-3.11.4/lib/python3.11/site-packages/torch_npu/utils/_path_manager.py:66: UserWarning: Permission mismatch: The owner of /opt/buildtools/Python-3.11.4/lib/python3.11/site-packages/torch_npu/lib/libop_plugin_atb.so does not match.
  warnings.warn(f"Permission mismatch: The owner of {path} does not match.")
/opt/buildtools/Python-3.11.4/lib/python3.11/site-packages/torch_npu/__init__.py:324: UserWarning: On the interactive interface, the value of TASK_QUEUE_ENABLE is set to 0 by default.                      Do not set it to 1 to prevent some unknown errors
  warnings.warn("On the interactive interface, the value of TASK_QUEUE_ENABLE is set to 0 by default. \
The attention mask is not set and cannot be inferred fr

Time cost:  5.205303502082825 ms
昇腾CANN（中国高性能计算机网络协会）的计算架构是基于一种叫做“Cann”的技术，它是一种基于量子计算原理的分布式计算框架。它的主要特点是：

1. 大规模并行计算：昇腾CANN支持大规模并行计算，可以有效地处理大量的数据。

2. 异步运算：昇腾CANN采用异步计算技术，可以实现快速响应用户请求，并在等待期间将请求交给其他进程。

3. 高效性：昇腾CANN采用了高效的数据结构和算法，可以在短时间内完成大量计算任务。

4. 低功耗：昇腾CANN利用量子计算的优势，可以实现更高的效率，同时还可以减少能源消耗。

5. 自动化运维：昇腾CANN提供了自动化运维功能，可以确保系统的稳定运行。


**（4）验证离线推理功能**

在代码中将模型路径改为本地绝对路径，即可实现离线启动（无需网络）。通过断开网络连接后重新运行推理，验证模型可在完全离线的环境下正常工作。

### 步骤六：基于AscendCL的推理应用开发

**（1）加载模型并转换为ONNX文件**

编写并执行以下代码：

```python
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

import torch
from transformers import AutoTokenizer, AutoModel
import onnx
from onnxruntime.quantization import quantize_dynamic, QuantType

if __name__ == "__main__":
    model_name = "Qwen/Qwen1.5-0.5B-Chat"
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    model = AutoModel.from_pretrained(model_name, torch_dtype=torch.float32, device_map="auto", trust_remote_code=True)
    
    model.eval()

    sample_input = "你好，请介绍一下你自己"
    inputs = tokenizer(sample_input, return_tensors="pt")
    
    input_names = ["input_ids", "attention_mask"]
    output_names = ["logits"]
    dynamic_axes = {
        "input_ids": {0: "batch_size", 1: "sequence_length"},
        "attention_mask": {0: "batch_size", 1: "sequence_length"},
        "logits": {0: "batch_size", 1: "sequence_length"}
    }

    torch.onnx.export(
        model,
        (inputs["input_ids"], inputs["attention_mask"]),
        "qwen15_05b_chat.onnx",
        export_params=True,
        opset_version=14,
        do_constant_folding=True,
        input_names=input_names,
        output_names=output_names,
        dynamic_axes=dynamic_axes,
        verbose=False,
    )

    print("ONNX export successful!")
```

In [1]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

import torch
from transformers import AutoModel, AutoTokenizer

model_name = "Qwen/Qwen1.5-0.5B-Chat"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.eval()

dummy_input = tokenizer("Hello, how are you?", return_tensors="pt")

torch.onnx.export(
    model,
    tuple(dummy_input.values()),
    "/home/developer/experiment/lab_09/qwen15_05b_chat.onnx",
    input_names=['input_ids', 'attention_mask'],
    output_names=['last_hidden_state'],
    dynamic_axes={  # 设置动态轴以支持可变长度
        'input_ids': {0: 'batch_size', 1: 'sequence_length'},
        'attention_mask': {0: 'batch_size', 1: 'sequence_length'},
    },
    opset_version=17
)

/opt/buildtools/Python-3.11.4/lib/python3.11/site-packages/torch_npu/utils/_path_manager.py:66: UserWarning: Permission mismatch: The owner of /opt/buildtools/Python-3.11.4/lib/python3.11/site-packages/torch_npu/lib/libop_plugin_atb.so does not match.
  warnings.warn(f"Permission mismatch: The owner of {path} does not match.")
/opt/buildtools/Python-3.11.4/lib/python3.11/site-packages/torch_npu/__init__.py:324: UserWarning: On the interactive interface, the value of TASK_QUEUE_ENABLE is set to 0 by default.                      Do not set it to 1 to prevent some unknown errors
  warnings.warn("On the interactive interface, the value of TASK_QUEUE_ENABLE is set to 0 by default. \
/home/developer/.local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/developer/.local/lib/python3.11/site-packages/tra

**（2）执行模型转换**

使用atc命令将ONNX模型转换为昇腾离线模型：

```bash
atc --model=qwen15_05b_chat.onnx \
    --framework=5 \
    --output=qwen15_05b_chat_fp16 \
    --input_format=ND \
    --input_shape="input_ids:1,16; attention_mask:1,16" \
    --soc_version=Ascend310B4 \
    --precision_mode_v2=fp16
```

In [7]:
!atc --model=/home/developer/experiment/lab_09/qwen15_05b_chat.onnx --framework=5 --output=/home/developer/experiment/lab_09/qwen15_05b_chat_fp16 --input_format=ND --input_shape="input_ids:1,16; attention_mask:1,16" --soc_version=Ascend910B3 --precision_mode_v2=fp16

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


ATC start working now, please wait for a moment.
.....
ATC run success, welcome to the next use.



**（3）CPU 平台部署**

In [2]:
import time
import numpy as np 
from transformers import AutoTokenizer
import onnxruntime as ort

class QwenONNXRuntime:
    def __init__(self, model_path):
        self.session = ort.InferenceSession(model_path, providers=["CPUExecutionProvider"])
        self.tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen1.5-0.5B-Chat", trust_remote_code=True)

    def generate(self, prompt, max_length=50):
        inputs = self.tokenizer(prompt, return_tensors="np")
        onnx_inputs = {
            "input_ids": inputs["input_ids"].astype(np.int64),
            "attention_mask": inputs["attention_mask"].astype(np.int64)
        }

        outputs = self.session.run(None, onnx_inputs)
        logits = outputs[0]

        predicted_token_id = np.argmax(logits[0, -1, :], axis=-1)

        return self.tokenizer.decode([predicted_token_id])

MODEL_PATH = "/home/developer/experiment/lab_09/qwen15_05b_chat.onnx"
onnx_model = QwenONNXRuntime(MODEL_PATH)

start_time = time.time()
result = onnx_model.generate("Hello, how are you")
end_time = time.time()
print("Time cost: ", (end_time - start_time) * 100 / 1000, "ms")
print(result)


2026-08-22 08:26:06.252607475 [W:onnxruntime:Default, device_discovery.cc:134 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/pci0000:00/0000:00:01.0/0000:01:00.0/0000:02:04.0/virtio2" because filename "virtio2" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


Time cost:  0.0071258544921875 ms
ber


**（4）使用PyACL或AscendCL运行推理**

AscendCL是昇腾计算开放编程框架，提供Device管理、Context管理、模型加载与执行等API，支持C/C++和Python编程语言。以下以伪代码形式展示使用AscendCL开发推理应用的核心流程：

```
// 1. AscendCL初始化
acl.init(config_path)

// 2. 运行管理资源申请
acl.rt.set_device(device_id)
context = acl.rt.create_context(device_id)

// 3. 模型加载
model_id = acl.mdl.load_from_file("model.om")
model_desc = acl.mdl.create_desc()
acl.mdl.get_desc(model_desc, model_id)

// 4. 数据预处理（可使用DVPP/AIPP）
// 对输入数据进行缩放、色域转换、归一化等处理

// 5. 创建输入输出Dataset
input_dataset = acl.mdl.create_dataset()
output_dataset = acl.mdl.create_dataset()

// 6. 执行模型推理
acl.mdl.execute(model_id, input_dataset, output_dataset)

// 7. 获取推理结果
// 从output_dataset中解析模型输出

// 8. 资源释放
acl.mdl.unload(model_id)
acl.rt.destroy_context(context)
acl.rt.reset_device(device_id)
acl.finalize()
```

AscendCL的推理应用开发流程清晰有序：先初始化AscendCL内部资源，再申请运行管理资源（如计算设备），然后加载离线模型、准备输入数据、执行推理获取结果，最后释放资源并去初始化。

In [8]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
os.environ["OMP_NUM_THREADS"] = "1"

import numpy as np 
from transformers import AutoTokenizer
import time
import acl

ACL_MEM_MALLOC_HUGE_FIRST = 0
ACL_MEMCPY_HOST_TO_DEVICE = 1
ACL_MEMCPY_DEVICE_TO_HOST = 2

MODEL_PATH = "/home/developer/exeriment/lab_09/qwen15_05b_chat_fp16.om"

MAX_LEN = 16
MAX_NEW_TOKENS = 30
eos_token_id = 151645
pad_token_id = 151643

def run_npu_inference(model_desc, model_id, input_ids, attention_mask):
    # 4. 创建输入数据集
    input_dataset = acl.mdl.create_dataset()
    
    input_ids_size = input_ids.nbytes
    print("input_ids_size = ", input_ids_size)
    input_ids_buffer, ret = acl.rt.malloc(input_ids_size, ACL_MEM_MALLOC_HUGE_FIRST)
    src_ids_ptr = acl.util.bytes_to_ptr(input_ids.tobytes())
    ret = acl.rt.memcpy(input_ids_buffer, input_ids_size,
                        src_ids_ptr, input_ids_size,
                        ACL_MEMCPY_HOST_TO_DEVICE)
    input_ids_data_buffer = acl.create_data_buffer(input_ids_buffer, input_ids_size)
    _, ret = acl.mdl.add_dataset_buffer(input_dataset, input_ids_data_buffer)

    input_mask_size = attention_mask.nbytes
    input_mask_buffer, ret = acl.rt.malloc(input_mask_size, ACL_MEM_MALLOC_HUGE_FIRST)
    src_mask_ptr = acl.util.bytes_to_ptr(attention_mask.tobytes())
    ret = acl.rt.memcpy(input_mask_buffer, input_mask_size,
                        src_mask_ptr, input_mask_size,
                        ACL_MEMCPY_HOST_TO_DEVICE)
    input_mask_data_buffer = acl.create_data_buffer(input_mask_buffer, input_mask_size)
    _, ret = acl.mdl.add_dataset_buffer(input_dataset, input_mask_data_buffer)

    # 5. 准备输出数据集
    output_size = acl.mdl.get_output_size_by_index(model_desc, 0)
    if output_size is None or output_size == 0:
        output_size = 128
    print("output_size = ", output_size)
    output_buffer, ret = acl.rt.malloc(output_size, ACL_MEM_MALLOC_HUGE_FIRST)
    output_dataset = acl.mdl.create_dataset()
    output_data_buffer = acl.create_data_buffer(output_buffer, output_size)
    _, ret = acl.mdl.add_dataset_buffer(output_dataset, output_data_buffer)

    # 6. 执行推理
    ret = acl.mdl.execute(model_id, input_dataset, output_dataset)
    # assert ret == 0, "Run execute失败"
    
    # 7. 获取输出结果
    result_buffer, ret = acl.rt.malloc_host(output_size)
    ret = acl.rt.memcpy(result_buffer, output_size,
                        output_buffer, output_size,
                        ACL_MEMCPY_DEVICE_TO_HOST)

    result_bytes = acl.util.ptr_to_bytes(result_buffer, output_size)
    result = np.frombuffer(result_bytes, dtype=np.int64)
        
    # 8. 释放资源
    acl.mdl.destroy_dataset(output_dataset)
    acl.mdl.destroy_dataset(input_dataset)
    acl.rt.free(output_buffer)
    acl.rt.free(input_ids_buffer)
    acl.rt.free(input_mask_buffer)
    acl.destroy_data_buffer(input_ids_data_buffer)
    acl.destroy_data_buffer(input_mask_data_buffer)
    acl.destroy_data_buffer(output_data_buffer)
    
    return result


In [9]:
from tokenizers import  Tokenizer
import time

model_name = "Qwen/Qwen1.5-0.5B-Chat"
tokenizer = Tokenizer.from_pretrained(model_name)

prompt = "Hello, how are you"
print("\nPrompt: ")
print(prompt)
    
# 正确的tokenization
encoding = tokenizer.encode(prompt)
input_tokens = encoding.ids
print(f"原始token IDs: {input_tokens}", len(input_tokens))
    
if len(input_tokens) > MAX_LEN:
    input_tokens = input_tokens[-MAX_LEN:]

valid_len = len(input_tokens)
pad_len = MAX_LEN - valid_len
print(f"valid_len = {valid_len}, pad_len = {pad_len}")

input_ids = np.array([[pad_token_id] * pad_len + input_tokens], dtype=np.int64)
attention_mask = np.array([[0] * pad_len + [1] * valid_len], dtype=np.int64)
print("\ninput_ids: ", input_ids, input_ids.shape)
print("\nattention_mask: ", attention_mask, attention_mask.shape)
    
print("\nGenerating...\n")
start_time = time.time()

# 1. ACL初始化
ret = acl.init()
assert ret == 0, "ACL初始化失败"

# 2. 打开设备并创建Context
device_id = 0
ret = acl.rt.set_device(device_id)
context, ret = acl.rt.create_context(device_id)
assert ret == 0, "创建Context失败"

# 3. 加载模型
model_id, ret = acl.mdl.load_from_file(MODEL_PATH)
model_desc = acl.mdl.create_desc()
ret = acl.mdl.get_desc(model_desc, model_id)

generated = []
for step in range(MAX_NEW_TOKENS):
    outputs = run_npu_inference(model_desc, model_id, input_ids, attention_mask)
    if outputs is None or outputs.size == 0:
        print("Inference returned empty, stopping generation.")
        continue

    print("outputs: ", outputs)
    logits = outputs
    next_token = int(np.argmax(logits))
    if next_token == eos_token_id:
        print("\n<eos>")
        break
    
    generated.append(next_token)
    
    input_ids = np.append(input_ids, [[next_token]], axis=1)
    attention_mask = np.append(attention_mask, [[1]], axis=1)
    print(tokenizer.decode([next_token]), end="", flush=True)
        
acl.mdl.destroy_desc(model_desc)
acl.mdl.unload(model_id)
acl.rt.destroy_context(context)
acl.rt.reset_device(device_id)
acl.finalize()
print("\n\n✅ Done.")

end_time = time.time()
print("Time cost: ", (end_time - start_time) * 100 / 1000, "ms")

print("The generate is: ", len(generated), generated)
text = np.array(generated)
answer = tokenizer.decode(text)
print("The generate text is: ", answer)


Prompt: 
Hello, how are you
原始token IDs: [9707, 11, 1246, 525, 498] 5
valid_len = 5, pad_len = 11

input_ids:  [[151643 151643 151643 151643 151643 151643 151643 151643 151643 151643
  151643   9707     11   1246    525    498]] (1, 16)

attention_mask:  [[0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1]] (1, 16)

Generating...

input_ids_size =  128
output_size =  128
outputs:  [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
!input_ids_size =  136
output_size =  128
outputs:  [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
!input_ids_size =  144
output_size =  128
outputs:  [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
!input_ids_size =  152
output_size =  128
outputs:  [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
!input_ids_size =  160
output_size =  128
outputs:  [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
!input_ids_size =  168
output_size =  128
outputs:  [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
!input_ids_size =  176
output_size =  128
outputs:  [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
!input_ids_size =  184
output_size =  128
outputs:  [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 

### 步骤七：结果验证与测试

（1）**CANN环境验证**：使用`npu-smi info`命令查看NPU状态，确认芯片型号、内存使用和温度等参数正常。

（2）**模型转换验证**：确认目标路径下生成了正确的.om离线模型文件。

（3）**推理功能验证**：通过多轮对话测试，观察模型回复的准确性和响应速度，确认大模型离线推理功能正常。

（4）**性能观察**：使用`npu-smi info watch`命令实时监控推理过程中NPU的资源占用情况。


## 六、任务拓展

学有余力的同学可进一步探索以下方向：

1. **模型量化压缩**：尝试将大模型进行INT8量化，降低模型存储和计算开销，对比量化前后推理精度和性能的变化，理解量化技术在资源受限设备上的重要意义。

2. **ATC动态输入配置**：探索ATC工具的动态Batch Size和动态分辨率功能，使得同一离线模型能够适配不同尺寸的输入数据，提升模型部署的灵活性。

3. **多框架适配对比**：分别使用MindSpore和PyTorch导出同一模型，通过ATC转换为离线模型后进行推理性能对比，分析不同框架在昇腾平台上的适配效率差异。

4. **AscendCL自定义应用开发**：基于AscendCL接口编写完整的推理应用，实现图像分类或目标检测等实用功能，深入理解昇腾推理应用开发的完整链路。

5. **vLLM推理服务部署**：在具备条件的设备上尝试部署vLLM-Ascend推理框架，体验高性能批量推理服务。


## 七、实验总结

通过本次实验，完成了基于香橙派AIpro开发板的CANN开发环境搭建与大模型离线推理的全流程实践。实验涵盖系统镜像烧录、CANN软件安装、环境变量配置、ATC模型转换以及大模型推理应用开发等关键环节。

**实验收获**：

1. **掌握了CANN环境搭建能力**：深入理解了CANN作为昇腾AI异构计算架构在AI处理器与上层框架之间的桥梁作用，学会了镜像烧录、软件包安装和环境变量配置的完整流程。
2. **理解了模型离线转换技术**：通过ATC工具的使用，理解了原始模型到离线模型的转换机制，掌握了atc命令的核心参数和配置方法。
3. **体验了端侧大模型推理**：成功在香橙派AIpro上运行了Qwen-1.5-0.5B大语言模型，体验了大模型在资源受限的边缘设备上进行离线推理的完整流程，这对于AIoT、智能机器人等离线场景具有重要意义。



**展望**：

边缘AI是人工智能发展的重要方向，昇腾CANN的全栈开放能力为国产AI生态建设提供了坚实基础。随着CANN架构的持续开源开放和大模型蒸馏技术的不断进步，未来在端侧设备上部署更强能力的大模型将成为现实，为智能安防、智能家居、自动驾驶等领域带来更广阔的应用前景。